In [ ]:
# Set thread environment variables for numerical stability
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

In [ ]:
# Add parent directory to Python path to find scripts module
import sys
from pathlib import Path

# Get the workspace root
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

In [ ]:
# Standard library imports
import copy
import pickle
import sys

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
from scripts.data_utils import load_nrm

# Local package imports - clustering
from scripts.clustering import core as sc
from scripts.clustering import shuffling as sh
#from scripts.clustering import evaluation_helpers as eval_helpers
#from scripts.clustering import distances as dist_helpers

# in distance.py, import of editdistance module is commented out

# Local package imports - visualization
from scripts.visualization import plots_clustering as pl
from scripts.visualization import plots_analysis as pa
from scripts.visualization import plots_simulation as ps
from scripts.visualization.style import set_plot_style, PublicationStandard

# Local package imports - analysis
from scripts.analysis import analysis as an

In [ ]:
set_plot_style()

In [ ]:
#load data
sig_range = (1,1)
vol_range = (0.1, 0.3)

with open(f"simulation_N100_sig_{sig_range}_vol_{vol_range}_rhosig_0.0_rhovol_0.0.pkl", "rb") as f:
    sim1 = pickle.load(f)

In [ ]:
with open(f"downsample_R100_N100000_sig_{sig_range}_vol_{vol_range}_rhosig_0.0_rhovol_0.0.pkl", "rb") as f:
    sim1 = pickle.load(f)

In [ ]:
# Load the survival scores for the downsampled data with the same parameters
with open(f"survival_scores_downsampled_{sig_range}_{vol_range}.pkl", "rb") as f:
    scores = pickle.load(f)

In [ ]:
seqs_labels=sim1['seqs_labels']
data = {
    "bursts": sim1['spike_times'],
    "seqs": sim1['seqs'],
    "seq_method": 'center_of_mass',
}

In [ ]:
# flatten the seqs from dict (values are lists) to a single flat list FOR DOWNSAMPLED DATA
seqs_list = []
for seqs in data['seqs'].values():
    seqs_list.extend(seqs)
data['seqs'] = seqs_list

In [ ]:
# survival score
scores = sh.survival_scores(
    data, k=50, thr_c=0.4,
    seeds=range(30)
)

In [ ]:
# save the survival scores for the downsampled data 
with open(f"survival_scores_downsampled_{sig_range}_{vol_range}.pkl", "wb") as f:
    pickle.dump(scores, f)

In [ ]:
# save the survival scores for the N=100 simulation data
with open(f"survival_scores_simulation_N100_{sig_range}_{vol_range}.pkl", "wb") as f:
    pickle.dump(scores, f)

In [ ]:
mask = (
    (scores["survival_freq"] > 0.1) &
    (scores["pairwise_jaccard_cond_survival"] > 0.1))

In [ ]:
pa.plot_survival_scores(
    scores,
    seqs_labels,
    "motif label",
    0.1, 0.1,
    figsize=(5,4),
    s=1,
    cmap="tab20"
)
#plt.savefig(f"survival_scores_downsampled_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")
plt.savefig(f"survival_scores_orig_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")

In [ ]:
# clustering

In [ ]:
mat_dict = sc.allmot(data["seqs"])
red, rd = sh.run_one(data["seqs"], data, mat_dict, k=30, thr_c=0.4)
ids_order = rd["ids_clust"].copy()
ids_order[mask] = -2
rd_ = sc.info_cluster(data["bursts"], data["seqs"], ids_order, method='center_of_mass')
sc.add_within_clust_score(rd_, mat_dict)
red_ = an.sort_and_filter_labels(
    ids_clust=rd_["ids_clust"],
    clust_scores=rd_["clust_scores"],
    sort_by="within_clust",
    ascending=True,
    min_score={"within_clust": 0.4, "within_clust_ratio": 0},
    min_size=1,
    replace_with=-1,
)
s = np.unique(red_["ids_clust_replaced"], return_counts=True)
sf_clust = s[1][np.where(s[0]>0)[0]]
count_seqs_real= sf_clust.sum()
count_clust_real= len(sf_clust)
print(s)

In [ ]:
fig, _, _ = pl.plot_mats(red_, mat_dict["zmat"])
#plt.savefig(f"cluster_mats_downsampled_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")
plt.savefig(f"cluster_mats_orig_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")

In [ ]:
tbl = ps.plot_cluster_composition(sim1['seqs_labels'], rd_["ids_clust"], normalize=False, sort="size", include_noise=True, 
                                  return_table=True, figsize=(8,4), ratios=rd_["clust_scores"]["within_clust"])
#plt.savefig(f"cluster_composition_downsampled_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")
plt.savefig(f"cluster_composition_orig_{sig_range}_{vol_range}.jpg", dpi=300, bbox_inches="tight")